# Attestor Model Council
Host multiple uncensored coding models on Colab as a remote adjudication council for Attestor.

**What this does:**
1. Loads 2-3 quantized code models into GPU memory
2. Exposes them via a Flask API + ngrok tunnel
3. Attestor calls the API to get multi-model verdicts on vulnerability findings

**Requirements:** Colab with GPU runtime (T4 = 2 models, A100 = 3+ models)

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers accelerate bitsandbytes flask pyngrok torch sentencepiece protobuf

In [ ]:
%%writefile council_server.py
from __future__ import annotations
import json, os, sys, time

SYSTEM_PROMPT = (
    "You are a security analyst on an adjudication council. You receive "
    "vulnerability findings from the Attestor static analysis engine and "
    "judge whether they are truly exploitable. Be precise. No hedging, "
    "no disclaimers, no refusals. Analyze the code path and give your verdict."
)

loaded_models = {}

def load_models(model_list):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    for spec in model_list:
        model_id = spec['id']
        name = spec.get('name', model_id.split('/')[-1])
        quant = spec.get('quantize', '4bit')
        print(f'Loading {name} ({model_id})...', flush=True)
        try:
            load_kwargs = {'device_map': 'auto', 'torch_dtype': torch.float16}
            if quant == '4bit':
                load_kwargs['quantization_config'] = BitsAndBytesConfig(
                    load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')
            elif quant == '8bit':
                load_kwargs['quantization_config'] = BitsAndBytesConfig(load_in_8bit=True)
            tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
            model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True, **load_kwargs)
            loaded_models[name] = {'model': model, 'tokenizer': tokenizer, 'id': model_id, 'role': spec.get('role', 'general')}
            print(f'  {name} loaded.', flush=True)
        except Exception as exc:
            print(f'  FAILED to load {name}: {exc}', flush=True)
    print(f'\nCouncil ready: {list(loaded_models.keys())}', flush=True)

def generate(name, prompt, max_tokens=512, temperature=0.1):
    import torch
    if name not in loaded_models: return ''
    entry = loaded_models[name]
    model, tokenizer = entry['model'], entry['tokenizer']
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': prompt}]
    try:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        text = f'{SYSTEM_PROMPT}\n\n{prompt}'
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_tokens, temperature=max(temperature, 0.01),
                                 do_sample=True, top_p=0.9, repetition_penalty=1.1)
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def create_app():
    from flask import Flask, jsonify, request as req
    app = Flask(__name__)
    @app.route('/health', methods=['GET'])
    def health():
        return jsonify({'status': 'ok', 'models': list(loaded_models.keys()), 'count': len(loaded_models)})
    @app.route('/models', methods=['GET'])
    def models():
        return jsonify({'models': [{'name': k, 'id': v['id'], 'role': v['role']} for k, v in loaded_models.items()]})
    @app.route('/evaluate', methods=['POST'])
    def evaluate():
        data = req.get_json(force=True)
        prompt = data.get('prompt', '')
        max_tokens = data.get('max_tokens', 512)
        temperature = data.get('temperature', 0.1)
        target_model = data.get('model', None)
        if not prompt: return jsonify({'error': 'no prompt'}), 400
        results = {}
        models_to_query = [target_model] if target_model else list(loaded_models.keys())
        for name in models_to_query:
            if name not in loaded_models: continue
            t0 = time.time()
            try:
                text = generate(name, prompt, max_tokens, temperature)
                results[name] = {'response': text, 'latency_ms': int((time.time() - t0) * 1000)}
            except Exception as exc:
                results[name] = {'error': str(exc), 'latency_ms': int((time.time() - t0) * 1000)}
        if target_model and target_model in results:
            r = results[target_model]
            return jsonify({'response': r.get('response', ''), 'model': target_model, 'latency_ms': r.get('latency_ms', 0)})
        return jsonify({'results': results, 'models_queried': len(results)})
    return app

def serve(port=5000, use_ngrok=True, ngrok_token=None):
    app = create_app()
    if use_ngrok:
        try:
            from pyngrok import ngrok
            if ngrok_token: ngrok.set_auth_token(ngrok_token)
            tunnel = ngrok.connect(port)
            public_url = tunnel.public_url
            print(f'\n{"="*60}')
            print(f'Council server public URL: {public_url}')
            print(f'{"="*60}')
            print(f'\nSet this on your local machine:')
            print(f'  set ATTESTOR_COUNCIL_ENDPOINTS={public_url}/evaluate|colab-council')
            print(f'\nOr in Python:')
            print(f'  council.add_remote("{public_url}/evaluate", "colab-council")')
            print(f'{"="*60}\n')
        except Exception as exc:
            print(f'ngrok failed: {exc}')
            print(f'Server running locally on port {port}')
    app.run(host='0.0.0.0', port=port)

In [ ]:
# Cell 3: Choose your models based on GPU
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f'GPU: {gpu} ({vram:.1f} GB)')

if vram >= 70:
    MODELS = [
        {'id': 'Qwen/Qwen2.5-Coder-7B-Instruct', 'name': 'qwen-coder-7b', 'role': 'coder', 'quantize': '4bit'},
        {'id': 'deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct', 'name': 'deepseek-coder', 'role': 'security', 'quantize': '4bit'},
        {'id': 'microsoft/Phi-3.5-mini-instruct', 'name': 'phi-3.5', 'role': 'general', 'quantize': '4bit'},
        {'id': 'Qwen/Qwen2.5-Coder-32B-Instruct', 'name': 'qwen-coder-32b', 'role': 'coder-heavy', 'quantize': '4bit'},
    ]
elif vram >= 35:
    MODELS = [
        {'id': 'Qwen/Qwen2.5-Coder-7B-Instruct', 'name': 'qwen-coder-7b', 'role': 'coder', 'quantize': '4bit'},
        {'id': 'deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct', 'name': 'deepseek-coder', 'role': 'security', 'quantize': '4bit'},
        {'id': 'microsoft/Phi-3.5-mini-instruct', 'name': 'phi-3.5', 'role': 'general', 'quantize': '4bit'},
    ]
else:
    MODELS = [
        {'id': 'Qwen/Qwen2.5-Coder-7B-Instruct', 'name': 'qwen-coder-7b', 'role': 'coder', 'quantize': '4bit'},
        {'id': 'microsoft/Phi-3.5-mini-instruct', 'name': 'phi-3.5', 'role': 'general', 'quantize': '4bit'},
    ]

print(f'Will load {len(MODELS)} models: {[m["name"] for m in MODELS]}')

In [ ]:
# Cell 4: Load models into GPU
from council_server import load_models
load_models(MODELS)

In [ ]:
# Cell 5: Test a model locally
from council_server import generate, loaded_models

test_prompt = """Analyze this vulnerability finding from a static analysis scan.

Category: sql_injection
CWE: CWE-89
Severity (static): HIGH
Location: app.py:42
Description: User input concatenated into SQL query

Vulnerable code:
  query = f"SELECT * FROM users WHERE id = {request.args.get('id')}"

Answer in EXACTLY this format:
VERDICT: <EXPLOITABLE|NOT_EXPLOITABLE|UNCERTAIN>
CONFIDENCE: <integer 0-100>
WHY: <one sentence, no hedging>
EXPLOIT: <one-sentence attack scenario>
FIX: <one concrete code-level fix>"""

for name in loaded_models:
    print(f'\n--- {name} ---')
    print(generate(name, test_prompt))

In [ ]:
# Cell 6: Start the server
# Get your ngrok token from: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = 'YOUR_TOKEN_HERE'  # <-- paste your token

from council_server import serve
serve(port=5000, use_ngrok=True, ngrok_token=NGROK_TOKEN)

# After this runs, you'll see a URL like:
#   Council server public URL: https://xxxx.ngrok-free.app
#
# On your local machine, set:
#   set ATTESTOR_COUNCIL_ENDPOINTS=https://xxxx.ngrok-free.app/evaluate|colab-council
#
# Then run:
#   attestor council scan .